In [0]:
df_silver = spark.table("ecommerce_electroics_sales.sales_schema.bronze_sales")
a=df_silver.count()
df_silver=df_silver.dropna(subset=["Order_ID","Order_Date","Price_Each"])
b=df_silver.count()
print("rows dropped", a-b)
df_silver.show()
print(b)

rows dropped 148
+--------+--------------------+----------------+----------+--------------+--------------------+
|Order_ID|             Product|Quantity_Ordered|Price_Each|    Order_Date|    Purchase_Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  176558|USB-C Charging Cable|               2|     11.95|04/19/19 08:46|917 1st St, Dalla...|
|  176559|Bose SoundSport H...|               1|     99.99|04/07/19 22:30|682 Chestnut St, ...|
|  176560|        Google Phone|               1|     600.0|04/12/19 14:38|669 Spruce St, Lo...|
|  176560|    Wired Headphones|               1|     11.99|04/12/19 14:38|669 Spruce St, Lo...|
|  176561|    Wired Headphones|               1|     11.99|04/30/19 09:27|333 8th St, Los A...|
|  176562|USB-C Charging Cable|               1|     11.95|04/29/19 13:03|381 Wilson St, Sa...|
|  176563|Bose SoundSport H...|               1|     99.99|04/02/19 07:46|668 Center St, Se...|
|  176564|USB-C Chargin

In [0]:
from pyspark.sql.functions import count, col

total = df_silver.count()
unique = df_silver.dropDuplicates(["Order_ID"]).count()

print("Total rows:", total)
print("Unique Order_IDs:", unique)
print("Duplicates:", total - unique)

Total rows: 30246
Unique Order_IDs: 29018
Duplicates: 1228


In [0]:
unique_combo = df_silver.dropDuplicates(["Order_ID", "Product"]).count()
print("Unique Order+Product:", unique_combo)
print("True duplicates:", total - unique_combo)

Unique Order+Product: 30198
True duplicates: 48


In [0]:
df_silver=df_silver.dropDuplicates(["Order_ID","Product"])
print("After dropping true duplicates",df_silver.count())

After dropping true duplicates 30198


In [0]:
df_silver.printSchema()

root
 |-- Order_ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity_Ordered: integer (nullable = true)
 |-- Price_Each: double (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Purchase_Address: string (nullable = true)



In [0]:
from pyspark.sql.functions import to_timestamp,col,month,hour,split,trim,sum,round

df_silver = df_silver.withColumn("Order_Date", to_timestamp(col("Order_Date"), "MM/dd/yy HH:mm"))

df_silver.printSchema()
df_silver.select("Order_Date").show(5)

root
 |-- Order_ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity_Ordered: integer (nullable = true)
 |-- Price_Each: double (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Purchase_Address: string (nullable = true)

+-------------------+
|         Order_Date|
+-------------------+
|2019-04-19 08:46:00|
|2019-04-07 22:30:00|
|2019-04-12 14:38:00|
|2019-04-12 14:38:00|
|2019-04-30 09:27:00|
+-------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver \
    .withColumn("Month", month(col("Order_Date"))) \
    .withColumn("Hour", hour(col("Order_Date"))) \
    .withColumn("City", trim(split(col("Purchase_Address"), ",")[1]))

df_silver.select("Order_Date", "Month", "Hour", "Purchase_Address", "City").show(5)

+-------------------+-----+----+--------------------+-----------+
|         Order_Date|Month|Hour|    Purchase_Address|       City|
+-------------------+-----+----+--------------------+-----------+
|2019-04-19 08:46:00|    4|   8|917 1st St, Dalla...|     Dallas|
|2019-04-07 22:30:00|    4|  22|682 Chestnut St, ...|     Boston|
|2019-04-12 14:38:00|    4|  14|669 Spruce St, Lo...|Los Angeles|
|2019-04-12 14:38:00|    4|  14|669 Spruce St, Lo...|Los Angeles|
|2019-04-30 09:27:00|    4|   9|333 8th St, Los A...|Los Angeles|
+-------------------+-----+----+--------------------+-----------+
only showing top 5 rows


In [0]:
df_silver=df_silver.withColumn("Revenue",col("Quantity_Ordered")*col("Price_Each"))
df_silver.show(5)

+--------+--------------------+----------------+----------+-------------------+--------------------+-----+----+-----------+-------+
|Order_ID|             Product|Quantity_Ordered|Price_Each|         Order_Date|    Purchase_Address|Month|Hour|       City|Revenue|
+--------+--------------------+----------------+----------+-------------------+--------------------+-----+----+-----------+-------+
|  176558|USB-C Charging Cable|               2|     11.95|2019-04-19 08:46:00|917 1st St, Dalla...|    4|   8|     Dallas|   23.9|
|  176559|Bose SoundSport H...|               1|     99.99|2019-04-07 22:30:00|682 Chestnut St, ...|    4|  22|     Boston|  99.99|
|  176560|        Google Phone|               1|     600.0|2019-04-12 14:38:00|669 Spruce St, Lo...|    4|  14|Los Angeles|  600.0|
|  176560|    Wired Headphones|               1|     11.99|2019-04-12 14:38:00|669 Spruce St, Lo...|    4|  14|Los Angeles|  11.99|
|  176561|    Wired Headphones|               1|     11.99|2019-04-30 09:27:

In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_electroics_sales.sales_schema.silver_sales")